In [1]:
import sys, os
from pathlib import Path
from pprint import pprint
from copy import deepcopy

import numpy as np
import pandas as pd

import yaml
from tqdm import tqdm

BASE_DIR = Path.cwd().resolve().parent.parent.parent
SCRIPT_DIR = Path.cwd().resolve().parent
sys.path.append(str(BASE_DIR))
group_name = 'fidelity_exp'
EXP_DIR = SCRIPT_DIR / group_name
EXP_CFG_DIR = EXP_DIR / 'configs'
RES_DIR = EXP_DIR / 'results'
CFG_DIR = SCRIPT_DIR / 'configs'

from experiment_board.anomaly_detection_journal_exps.simulated_experiment_classes import Experiment, hp_study, SNN_LOGN_GTVObjective, get_data, calculate_metrics

In [2]:
def load_study_results(group_name, config_name, model_name):
    """
    Load study results from a CSV file.
    """
    file_path = SCRIPT_DIR / group_name / 'results' / f"{config_name}_{model_name}.csv"
    if not file_path.exists():
        raise FileNotFoundError(f"Results file {file_path} does not exist.")
    
    df = pd.read_csv(file_path)
    return df

def convert_angle_to_cartesian_on_simplex(thetas):
    """Convert hyperparameters in spherical coordinates (thetas) to Cartesian coordinates on N-simplex.
    
    Args:
        thetas (np.ndarray): Angles in radians, shape (-1,N-1) where N is the number of dimensions.
    Returns:
        np.ndarray: Cartesian coordinates on N-simplex, shape (-1,N).
    """
    dim_1 = thetas.shape[0]; dim_2 = thetas.shape[1]
    thetas = np.array(thetas, dtype=np.float64)
    sin_thetas = np.concatenate([np.ones((dim_1,1)), np.sin(thetas)], axis=1)
    cos_thetas = np.concatenate([np.cos(thetas), np.ones((dim_1,1))], axis=1)
    x = (np.cumprod(sin_thetas, axis=1)*cos_thetas)**2
    return x

def load_exp_config(group_name, config_name):
    """
    Load experiment configuration from a YAML file.
    """
    with open(EXP_CFG_DIR / f'{config_name}.yaml') as f:
        exp_config = yaml.safe_load(f)
    return exp_config

def set_nested_value(data, keys, value):
    """
    Sets a value in a nested dictionary given a list of keys.

    Args:
        data (dict): The dictionary to modify.
        keys (list): A list of keys representing the path to the value.
        value: The value to set.
    """
    if not keys:
        return value
    if not isinstance(data, dict):
      data = {}
    data[keys[0]] = set_nested_value(data.get(keys[0], {}), keys[1:], value)
    return data

In [15]:
df = load_study_results("fidelity_exp", "coherent_SL_err", 'HoRPCA')
df.head()


,X,X_t_rank,S,AUC-ROC,AUC-PRC,S_err,L_err,rel_err,primal_residual,dual_residual,...,theta_1,theta_2,theta_3,dummy,theta_2_importance,theta_3_importance,theta_1_importance,SNR,seed,model
0,270000.000000,99.666667,0.0,0.810481,0.329414,1.008325,0.231930,1.240256,0.095022,0.092796,...,0.965094,0.977515,1.198676,0,0.373626,0.329670,0.296703,20,31,HoRPCA
1,270000.000000,99.666667,0.0,0.870885,0.449501,0.990655,0.167608,1.158263,0.095800,0.013069,...,0.254020,1.173858,0.532309,0,0.363636,0.333333,0.303030,24,31,HoRPCA
2,270000.000000,99.666667,0.0,0.982316,0.891913,0.987939,0.122785,1.110724,0.079540,0.006956,...,0.200317,0.467755,0.719381,0,0.352941,0.352941,0.294118,32,31,HoRPCA
3,83203.333333,45.000000,199092.0,0.962736,0.836448,0.884560,0.078235,0.962795,0.125886,0.012617,...,1.416385,0.319934,0.991121,0,0.428571,0.380952,0.190476,28,31,HoRPCA
4,270000.000000,99.666667,0.0,0.520712,0.052144,1.000946,0.801696,1.802642,0.097292,0.023097,...,0.435838,0.599147,0.659904,0,0.451613,0.322581,0.225806,8,31,HoRPCA


In [3]:
group_name = "fidelity_exp"
config_name = "coherent_SL_err"
# Load the experiment configuration
exp_config = load_exp_config(group_name, config_name)
overwrite = True

independent_var = exp_config['independent_var']
model_configs = exp_config['models']
model_keys = list(model_configs.keys())
model_names = [model_configs[key]['name'] for key in model_keys]

aggregate_results = []
repeat = 10
for i, model_key in enumerate(model_keys):
    model_cfg = model_configs[model_key]
    model_name = model_cfg['name']
    print(f"Loading study results for model: {model_name}")
    df = load_study_results(group_name, config_name, model_name)
    model_aggregate_results = []
    for j in tqdm(range(len(independent_var['range'])), desc=f"Processing {model_name}"):
        # extract the row of the dataframe where the independent variable matches the current range value
        row = df[df[independent_var['keys'][-1]] == independent_var['range'][j]]
        if row.empty:
            print(f"No data found for independent variable range {independent_var['range'][j]}")
            continue
        # extract the theta values for the current range value
        thetas = np.array(
                    row[[col for col in row.columns if col.startswith('theta_') and not col.endswith('_importance')]].values
                    ).ravel()

        lda_gtvs = row[[col for col in row.columns if col.startswith('lda_gtv_') and not col.endswith('_importance')]].values.item()
        hyperparams = {'thetas': thetas, 'lda_gtvs': lda_gtvs}
        
        model_results = []
        print(f"Processing model {model_name} for {independent_var['keys'][-1]} = {independent_var['range'][j]}, {repeat} times")
        for k in range(repeat):
            data_var = set_nested_value(deepcopy(exp_config['data']),
                                        independent_var['keys'], independent_var['range'][j])
            data_var['repeated'] = False
            model_cfg['max_iter'] = 200
            model_cfg['device'] = 'cuda:2'
            data_var['seed'] = k
            obj = SNN_LOGN_GTVObjective(data_variables=data_var,study_config={},
                                model_config=model_cfg)
            res = obj.run_with_hyperparameters(hyperparams)
            res['seed'] = k
            res[independent_var['keys'][-1]] = independent_var['range'][j]
            model_results.append(res)

        model_result_path = RES_DIR / f"{config_name}_{model_name}_results.csv"
        model_result_df = pd.DataFrame(model_results)
        if os.path.exists(model_result_path) and not overwrite:
            model_result_df.to_csv(model_result_path, mode='a', header=False, index=False)
        else:
            model_result_df.to_csv(model_result_path, mode='w', header=True, index=False)

        model_mean_result = {kk: np.mean([m[kk] for m in model_results]) for kk in model_results[0].keys()}
        model_min_result = {'min_'+ kk: np.min([m[kk] for m in model_results]) for kk in model_results[0].keys()}
        model_max_result = {'max_'+ kk: np.max([m[kk] for m in model_results]) for kk in model_results[0].keys()}
        model_std_result = {'std_'+ kk: np.std([m[kk] for m in model_results]) for kk in model_results[0].keys()}
        model_aggregate_result = {**model_mean_result, **model_min_result, **model_max_result, **model_std_result}
        model_aggregate_result[independent_var['keys'][-1]] = independent_var['range'][j]
        model_aggregate_result['model_name'] = model_name
        model_aggregate_results.append(model_aggregate_result)
    aggregate_results += model_aggregate_results

aggregate_path = RES_DIR / f"{config_name}_{model_name}_aggregate_results.csv"
aggregate_df = pd.DataFrame(aggregate_results)
if os.path.exists(aggregate_path) and not overwrite:
    aggregate_df.to_csv(aggregate_path, mode='a', header=False, index=['model_name', independent_var['keys'][-1]])
else:
    aggregate_df.to_csv(aggregate_path, mode='w', header=True, index=['model_name', independent_var['keys'][-1]])


Loading study results for model: HoRPCA


Processing HoRPCA:   0%|          | 0/8 [00:00<?, ?it/s]

Processing model HoRPCA for SNR = 32, 10 times


/mnt/ffs24/home/indibimu/repos/ML_GSP/src/proximal_ops/prox_overlapping_grouped_l21.py:396: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  G_ind = torch.sparse.spdiags(diagonals= torch.ones(G.number_of_nodes(), device=device, dtype=dtype),
Processing HoRPCA:  12%|█▎        | 1/8 [00:32<03:50, 32.94s/it]

Processing model HoRPCA for SNR = 28, 10 times


...

Loading study results for model: [SNN]-[L1+GTV(S)]


Processing [SNN]-[L1+GTV(S)]:   0%|          | 0/8 [00:00<?, ?it/s]

Processing model [SNN]-[L1+GTV(S)] for SNR = 32, 10 times


Processing [SNN]-[L1+GTV(S)]:   0%|          | 0/8 [00:00<?, ?it/s]


TypeError: unsupported operand type(s) for *: 'numpy.ndarray' and 'Tensor'

In [10]:
# lda_gtvs
# row


0.0169216875314968